In [13]:
!pip install bertopic

In [14]:
import re
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import pandas as pd

df = pd.read_csv("quran_sahih.csv")
docs = df["text"].astype(str).tolist()
  # ===============================
# 1. Preprocessing
# ===============================
multi_stopwords = [
    "as for",
    "those who",
    "so that",
    "among whom",
    "amongst them",
    "o mankind",
    "o believers",
    "o people",
]

# --- NEW FUNCTION FOR GROUPING ---
def group_ayahs(ayahs_list, group_size=10):
    """Groups consecutive Ayahs (verses) into single documents."""
    segments = []
    # Iterate through the list, stepping by the group_size (10)
    for i in range(0, len(ayahs_list), group_size):
        # Slice the list to get the current segment
        segment = ayahs_list[i : i + group_size]
        # Join the Ayahs in the segment into one string document
        segments.append(" ".join(segment))
    return segments

def clean_text(doc):
    doc = str(doc).lower()
    for phrase in multi_stopwords:
        doc = doc.replace(phrase, " ")
    doc = re.sub(r"[^a-zA-Z\s]", " ", doc)
    tokens = doc.split()
    return " ".join(tokens)
AYAH_GROUP_SIZE = 10
docs_grouped = group_ayahs(docs, group_size=AYAH_GROUP_SIZE)
docs_clean = [clean_text(doc) for doc in docs_grouped]

# ===============================
# 2. STOPWORDS — USE ONLY ENGLISH
# ===============================
stop_words = [
    # Standard English stopwords
    "a","about","above","after","again","against","all","am","an","and","any","are",
    "as","at","be","because","been","before","being","below","between","both","but",
    "by","can","could","did","do","does","doing","down","during","each","few","for",
    "from","had","has","have","having","he","her","here","hers","herself","him",
    "himself","his","how","i","if","in","into","is","it","its","itself","just","me",
    "more","most","my","myself","no","nor","not","of","off","on","once","only","or",
    "other","our","ours","ourselves","out","over","own","same","she","should","so",
    "some","such","than","that","the","their","theirs","them","themselves","then",
    "there","these","they","this","those","through","to","too","under","until","up",
    "very","was","we","were","what","when","where","which","while","who","whom","why",
    "will","with","would","you","your","yours","yourself","yourselves", "is", "and",

    # Sahih-specific
    "indeed","then","thus","so","such","among","amongst","because","as for","so that",
    "those who","thereby","therein","thereafter","thereof","thereon","whereby",
    "wherein","whereof","whereupon","whatever","whichever","whomever","whenever",
    "wherever","anyone","anything","everyone","everything","either","neither",
    "whether", "upon", "us",

    # Quranic formulas
    "say","o","s",

    # Structural Quran translation particles
    "hereafter","thereafter","except",
    "lest"
]

# ===============================
# 3. VECTORIZER
# ===============================
vectorizer = CountVectorizer(
    stop_words=stop_words,
    lowercase=True,
    ngram_range=(1,2),   # unigram only
    min_df=2,
    max_df=0.80
)

# ===============================
# 4. UMAP
# ===============================
umap_model = UMAP(
    n_neighbors=50,
    n_components=15,
    min_dist=0.3,
    metric='cosine',
    random_state=42
)

# ===============================
# 5. HDBSCAN
# ===============================
hdbscan_model = HDBSCAN(
    min_cluster_size=5,
    min_samples=2,
    cluster_selection_method='leaf',
)

# ===============================
# 6. EMBEDDINGS – YOU MUST USE THIS
# ===============================
embedding_model = SentenceTransformer(
    "paraphrase-multilingual-mpnet-base-v2"
)

# ===============================
# 7. BERTopic
# ===============================
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics="auto"
)

topics, probs = topic_model.fit_transform(docs_clean)
topic_model.visualize_topics()


In [15]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,332,-1_fear_believe_believed_whoever,"[fear, believe, believed, whoever, truth, like...",[and do not approach the orphans property exce...
1,0,142,0_whoever_believed_believers_believe,"[whoever, believed, believers, believe, may, e...",[and if two factions among the believers shoul...
2,1,16,1_mountains_faces day_faces_dust,"[mountains, faces day, faces, dust, whoever, h...",[their assembly will be defeated and they will...
3,2,15,2_warn_arrogant_invite_scales,"[warn, arrogant, invite, scales, put, se, sign...",[or of have divided their religion and become ...
4,3,14,3_fruit_paradise_reclining_gardens,"[fruit, paradise, reclining, gardens, burn, pe...",[but feared their lord will be driven to parad...
5,4,13,4_moses_pharaoh_throw_magicians,"[moses, pharaoh, throw, magicians, pharaoh sai...",[pharaoh said if you take a god other than me ...
6,5,13,5_signs_created_night_forth,"[signs, created, night, forth, subjected, crea...",[it is he who sends down rain from the sky fro...
7,6,10,6_seized_destroyed_messengers_thamud,"[seized, destroyed, messengers, thamud, mercy,...",[as if they had never prospered therein unques...
8,7,10,7_abraham_gave_guided_moses,"[abraham, gave, guided, moses, descendants, is...",[and we caused the people who had been oppress...
9,8,7,8_known_drink_water_already known,"[known, drink, water, already known, seen, day...",[woe that day to the deniers did we not create...


In [16]:
new_topics = topic_model.reduce_outliers(documents = docs_clean, topics = topics, strategy="c-tf-idf")

# You MUST run this step to update the model's internal representations
topic_model.update_topics(docs_clean, topics = new_topics, vectorizer_model=vectorizer)

print("Outliers successfully reduced and reassigned using the c-TF-IDF strategy.")
print("\nUpdated Topic Information:")
# Check the count of the -1 topic to verify the reduction
(topic_model.get_topic_info())

2025-12-05 04:43:00,066 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers successfully reduced and reassigned using the c-TF-IDF strategy.

Updated Topic Information:


,Topic,Count,Name,Representation,Representative_Docs
0,0,368,0_believers_knowing_give_ever,"[believers, knowing, give, ever, revealed, evi...",[and if two factions among the believers shoul...
1,1,24,1_woe_recompense_faces day_mountains,"[woe, recompense, faces day, mountains, make k...",[their assembly will be defeated and they will...
2,2,18,2_arrogant_warn_better_invite,"[arrogant, warn, better, invite, scales, signs...",[or of have divided their religion and become ...
3,3,30,3_paradise_fruit_companions_hellfire,"[paradise, fruit, companions, hellfire, brough...",[but feared their lord will be driven to parad...
4,4,18,4_moses_pharaoh_throw_moses said,"[moses, pharaoh, throw, moses said, pharaoh sa...",[pharaoh said if you take a god other than me ...
5,5,29,5_signs_creation_rain_created man,"[signs, creation, rain, created man, sky, fort...",[it is he who sends down rain from the sky fro...
6,6,17,6_seized_lot_destroyed_thamud,"[seized, lot, destroyed, thamud, abraham, good...",[as if they had never prospered therein unques...
7,7,23,7_abraham_mention_gave_descendants,"[abraham, mention, gave, descendants, isaac, i...",[and we caused the people who had been oppress...
8,8,17,8_water_deniers_drink_eat,"[water, deniers, drink, eat, noah, day deniers...",[woe that day to the deniers did we not create...
9,9,8,9_paradise_companions_companions paradise_unjust,"[paradise, companions, companions paradise, un...",[so who is more unjust than he who invents a l...


In [17]:
# Assuming you want to sub-cluster Topic 3
TARGET_TOPIC_ID = 0

# 1. Get the indices of all documents belonging to the target topic
target_indices = [i for i, topic_id in enumerate(topics) if topic_id == TARGET_TOPIC_ID]

# 2. Extract the documents for the sub-clustering
sub_cluster_docs = [docs_clean[i] for i in target_indices]

print(f"Number of documents in Topic {TARGET_TOPIC_ID}: {len(sub_cluster_docs)}")
# 1. Create a fresh BERTopic model instance
sub_topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer,
    # Re-use your UMAP/HDBSCAN models or create new ones for tuning
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics="auto"
)

# 2. Fit and Transform the subset of documents
sub_topics, sub_probs = sub_topic_model.fit_transform(sub_cluster_docs)

Number of documents in Topic 0: 142


In [18]:
sub_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,20,-1_moses_fire_within_righteous,"[moses, fire, within, righteous, heavens, deed...",[allah is subtle with his servants he gives pr...
1,0,21,0_fire_ever_sin_righteous,"[fire, ever, sin, righteous, painful punishmen...",[so allah gave them the reward of this world a...
2,1,17,1_believers_land_treaty_establish,"[believers, land, treaty, establish, fear alla...",[and obey allah and obey the messenger but if ...
3,2,11,2_wives_ever_houses_believers,"[wives, ever, houses, believers, women, find, ...",[and women of post menstrual age who have no d...
4,3,10,3_adam_book_signs_created,"[adam, book, signs, created, heavens, exalted,...",[and mention o muhammad the brother of aad whe...
5,4,9,4_deity_exalted_heavens_heavens earth,"[deity, exalted, heavens, heavens earth, belon...",[to allah belongs whatever is in the heavens a...
6,5,9,5_said people_family_command_bring,"[said people, family, command, bring, solomon,...",[and his people came hastening to him and befo...
7,6,9,6_judge_allah revealed_turn_turn away,"[judge, allah revealed, turn, turn away, ask, ...",[they swear to you you might be satisfied with...
8,7,7,7_boy_allah said_threw_iblees,"[boy, allah said, threw, iblees, prostrated, p...",[so he came out to his people from the prayer ...
9,8,7,8_created_equal_allah made_used,"[created, equal, allah made, used, need, knows...",[it is he who created you and among you is the...


In [19]:

new_sub_topics = sub_topic_model.reduce_outliers(documents = sub_cluster_docs, topics = sub_topics, strategy="c-tf-idf")

# You MUST run this step to update the model's internal representations
sub_topic_model.update_topics(sub_cluster_docs, topics = new_sub_topics, vectorizer_model = vectorizer)

print("Outliers successfully reduced and reassigned using the c-TF-IDF strategy.")
print("\nUpdated Topic Information:")
# Check the count of the -1 topic to verify the reduction
(sub_topic_model.get_topic_info())

2025-12-05 04:43:04,516 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers successfully reduced and reassigned using the c-TF-IDF strategy.

Updated Topic Information:


,Topic,Count,Name,Representation,Representative_Docs
0,0,25,0_righteous_deeds_world_sin,"[righteous, deeds, world, sin, painful punishm...",[so allah gave them the reward of this world a...
1,1,19,1_land_rely_fear allah_treaty,"[land, rely, fear allah, treaty, signs, establ...",[and obey allah and obey the messenger but if ...
2,2,13,2_women_wives_houses_believing,"[women, wives, houses, believing, ever allah, ...",[and women of post menstrual age who have no d...
3,3,15,3_heavens_adam_produce_moses,"[heavens, adam, produce, moses, book, created,...",[and mention o muhammad the brother of aad whe...
4,4,9,4_deity_heavens_heavens earth_belongs,"[deity, heavens, heavens earth, belongs, book,...",[to allah belongs whatever is in the heavens a...
5,5,10,5_said people_family_command_bring,"[said people, family, command, bring, solomon,...",[and his people came hastening to him and befo...
6,6,9,6_judge_allah revealed_turn_ask,"[judge, allah revealed, turn, ask, turn away, ...",[they swear to you you might be satisfied with...
7,7,9,7_mary_angels_prostrate_allah said,"[mary, angels, prostrate, allah said, threw, j...",[so he came out to his people from the prayer ...
8,8,8,8_created_heavens_need_heavens earth,"[created, heavens, need, heavens earth, free n...",[it is he who created you and among you is the...
9,9,9,9_moses_moses said_pharaoh_said lord,"[moses, moses said, pharaoh, said lord, allah ...",[and then you did your deed which you did and ...


In [21]:
(topic_model.get_topic_info())

,Topic,Count,Name,Representation,Representative_Docs
0,0,368,0_believers_knowing_give_ever,"[believers, knowing, give, ever, revealed, evi...",[and if two factions among the believers shoul...
1,1,24,1_woe_recompense_faces day_mountains,"[woe, recompense, faces day, mountains, make k...",[their assembly will be defeated and they will...
2,2,18,2_arrogant_warn_better_invite,"[arrogant, warn, better, invite, scales, signs...",[or of have divided their religion and become ...
3,3,30,3_paradise_fruit_companions_hellfire,"[paradise, fruit, companions, hellfire, brough...",[but feared their lord will be driven to parad...
4,4,18,4_moses_pharaoh_throw_moses said,"[moses, pharaoh, throw, moses said, pharaoh sa...",[pharaoh said if you take a god other than me ...
5,5,29,5_signs_creation_rain_created man,"[signs, creation, rain, created man, sky, fort...",[it is he who sends down rain from the sky fro...
6,6,17,6_seized_lot_destroyed_thamud,"[seized, lot, destroyed, thamud, abraham, good...",[as if they had never prospered therein unques...
7,7,23,7_abraham_mention_gave_descendants,"[abraham, mention, gave, descendants, isaac, i...",[and we caused the people who had been oppress...
8,8,17,8_water_deniers_drink_eat,"[water, deniers, drink, eat, noah, day deniers...",[woe that day to the deniers did we not create...
9,9,8,9_paradise_companions_companions paradise_unjust,"[paradise, companions, companions paradise, un...",[so who is more unjust than he who invents a l...


from matplotlib import pyplot as plt
_df_0['Topic'].plot(kind='hist', bins=20, title='Topic')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_1['Count'].plot(kind='hist', bins=20, title='Count')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_2.plot(kind='scatter', x='Topic', y='Count', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  xs = series['Topic']
  ys = series['Count']
  
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_3.sort_values('Topic', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('Topic')
_ = plt.ylabel('Count')

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['Topic']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'Topic'}, axis=1)
              .sort_values('Topic', ascending=True))
  xs = counted['Topic']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_4.sort_values('Topic', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('Topic')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
_df_5['Topic'].plot(kind='line', figsize=(8, 4), title='Topic')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_6['Count'].plot(kind='line', figsize=(8, 4), title='Count')
plt.gca().spines[['top', 'right']].set_visible(False)